<a href="https://colab.research.google.com/github/kalp121212/The_Whodunit_Engine/blob/main/CS7634_Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Gen AI SDK allows you to prompt LLMs hosted by Google. You need to create a Gemini API key to use this. The dashboard listing your API key will show you which tier the key is for - free or otherwise. The code snippet below connects you to latest Gemini Flash model. You can swap out model names for the variable 'model' to try the same prompt with different models. The list of model names can be found at https://ai.google.dev/gemini-api/docs/models.

---



In [1]:
from google import genai
import json
import random
import time
from google.api_core.exceptions import ServiceUnavailable
from config import API_KEY, MIN_ITERATIONS, MIN_WRONG_ACCUS, MIN_PLOT_POINTS, MAX_COHERENCE_RETRIES

#flash -> creates more diverse and interesting stories, takes time though
#flash-lite -> creates good stories but all are similar, much faster though
# You can play around with the models and see which one you like better. I have been using flash for all my testing and it has been working well, but you can try flash-lite if you want faster results.
model = "models/gemini-flash-latest"

client = genai.Client(api_key = API_KEY)

First, we define a simple retry mechanism for catching 503 errors while trying to generate content to ensure we do not have to keep rerunning when the model is in high demand

In [2]:
def retry_generate_content(client_obj, model_name, contents, config, max_retries=5, initial_delay=1.0):
    for i in range(max_retries):
        try:
            response = client_obj.models.generate_content(
                model=model_name,
                contents=contents,
                config=config
            )
            # If successful, return the response
            return response
        except Exception as e:
            if e.status_code != 503:  # If it's not a ServiceUnavailable error, re-raise immediately
                raise e
            delay = initial_delay * (2 ** i)  # Exponential backoff
            print(f"ServiceUnavailable error encountered. Retrying in {delay:.2f} seconds... (Attempt {i + 1}/{max_retries})")
            time.sleep(delay)
    # If all retries fail, raise the last ServiceUnavailable error
    raise ServiceUnavailable(f"Failed to generate content after {max_retries} retries.")

## Module 1: Generate Setting

In [3]:
def generate_setting(client, model):
  # prompt
  prompt = """
  Generate a JSON object describing the setting of a whodunit murder mystery story.

  Your output will be used for a story where a detective must figure out the murderer from multiple suspects using clues and red herrings.

  Include the following fields:

  "setting": the location(s) where the murder and investigation occur.

  "tone": the narrative style (choose from "psychological", "noir", "classic puzzle").

  "time_period": the era or year in which the story takes place.

  "constraints": a list of rules or limitations that make this a classic whodunit:
  - number of suspects present
  - location access restrictions
  - isolation from outside help
  - solvability mechanics (clues, hidden motives, red herrings)

  Return ONLY a JSON object with no extra text.

  Note: Ensure that the name of characters are uncommon and not classic names
  like Alaistar Finch and the setting is not the same as most books like a
  Victorian Manor, it should be diverse and new.

  """

  config = {
      "response_mime_type": "application/json",
      "response_schema": {
          "type": "object",
          "properties": {
              "setting": {
                  "type": "string"
              },
              "tone": {
                  "type": "string",
                  "enum": ["psychological", "noir", "classic puzzle"]
              },
              "time_period": {
                  "type": "string"
              },
              "constraints": {
                  "type": "array",
                  "items": {
                      "type": "string"
                  }
              }
          },
          "required": [
              "setting",
              "tone",
              "time_period",
              "constraints"
          ]
      }
    }

  # api call
  response = retry_generate_content(
      client_obj=client,
      model_name=model,
      contents=prompt,
      config=config
  )

  # Convert JSON string -> Python dict
  data = json.loads(response.text)
  return data

## Module 2: Generating Crime

In [4]:
def generate_crime(client, model, setting):
  # Assume this came from your previous module
  world_context = json.dumps(setting, indent=2)

  # -------- Crime Generator Prompt -------- #

  prompt = f"""
  Generate the TRUE hidden details of a murder in a whodunit mystery.

  This information represents the real events of the crime and will NOT be shown to the story generator directly.

  World Context:
  {world_context}

  Generate the following:

  murderer
  victim
  true_motive
  superficial_motive
  means_of_murder
  timeline
  mistake_made

  Rules:
  - The crime should appear impossible to solve at first glance.
  - The timeline must be logically consistent.
  - The murderer must make one subtle mistake that will later reveal the truth.

  Return ONLY JSON in the following format:

  {{
    "murderer": "",
    "victim": "",
    "true_motive": "",
    "superficial_motive": "",
    "means_of_murder": "",
    "timeline": [],
    "mistake_made": ""
  }}
  """

  # -------- Schema Config -------- #

  config = {
      "response_mime_type": "application/json",
      "response_schema": {
          "type": "object",
          "properties": {
              "murderer": {"type": "string"},
              "victim": {"type": "string"},
              "true_motive": {"type": "string"},
              "superficial_motive": {"type": "string"},
              "means_of_murder": {"type": "string"},
              "timeline": {
                  "type": "array",
                  "items": {"type": "string"}
              },
              "mistake_made": {"type": "string"}
          },
          "required": [
              "murderer",
              "victim",
              "true_motive",
              "superficial_motive",
              "means_of_murder",
              "timeline",
              "mistake_made"
          ]
      }
  }

  # -------- Gemini API Call -------- #

  response = retry_generate_content(
      client_obj=client,
      model_name=model,
      contents=prompt,
      config=config
  )

  data = json.loads(response.text)
  return data

## Module 3: Generating Characters

In [5]:
def generate_characters(client, model, setting, crime):
  world_context = json.dumps(setting, indent=2)

  murderer = crime["murderer"]
  victim = crime["victim"]

  prompt = f"""
  Generate the suspects for a whodunit murder mystery.

  WORLD CONTEXT:
  {world_context}

  CRIME CONTEXT:
  Murderer: {murderer}
  Victim: {victim}

  Generate exactly 5 suspects including the murderer.

  Each suspect must include:
  name
  relation_to_victim
  public_image
  private_secret
  possible_motive
  alibi
  is_murderer

  Rules:
  - Only one suspect is the murderer.
  - Each suspect has a secret.
  - At least two suspects should strongly appear guilty but are innocent.
  - The murderer must have a misleading alibi.
  - Ensure that the name of characters are uncommon and not classic names
  like Alaistar Finch

  Return ONLY JSON in this format:

  {{
    "suspects": [
      {{
        "name": "",
        "relation_to_victim": "",
        "public_image": "",
        "private_secret": "",
        "possible_motive": "",
        "alibi": "",
        "is_murderer": false
      }}
    ]
  }}
  """

  config = {
      "response_mime_type": "application/json",
      "response_schema": {
          "type": "object",
          "properties": {
              "suspects": {
                  "type": "array",
                  "items": {
                      "type": "object",
                      "properties": {
                          "name": {"type": "string"},
                          "relation_to_victim": {"type": "string"},
                          "public_image": {"type": "string"},
                          "private_secret": {"type": "string"},
                          "possible_motive": {"type": "string"},
                          "alibi": {"type": "string"},
                          "is_murderer": {"type": "boolean"}
                      },
                      "required": [
                          "name",
                          "relation_to_victim",
                          "public_image",
                          "private_secret",
                          "possible_motive",
                          "alibi",
                          "is_murderer"
                      ]
                  }
              }
          }
      }
  }

  response = retry_generate_content(
      client_obj=client,
      model_name=model,
      contents=prompt,
      config=config
  )

  data = json.loads(response.text)
  return data

## Module 4: Generating Clues

In [6]:
def generate_clues(client, model, setting, crime, suspects):
  # Pass previous module outputs
  world_context = json.dumps(setting, indent=2)
  crime_context = json.dumps(crime, indent=2)
  suspects_context = json.dumps(suspects, indent=2)

  prompt = f"""
  Generate the clues for a whodunit murder mystery.

  WORLD CONTEXT:
  {world_context}

  CRIME CONTEXT:
  {crime_context}

  SUSPECTS:
  {suspects_context}

  Your task is to generate:

  1. physical_clues
  2. psychological_clues
  3. misdirection_clues
  4. timeline_contradiction_clues

  Rules:
  - All clues must be consistent with the true crime.
  - At least 2 misdirection clues should point to innocent suspects.
  - Timeline contradiction clues should be subtle but solvable.
  - Clues should naturally integrate into the setting.
  - Return ONLY JSON.

  JSON format:
  {{
    "physical_clues": [],
    "psychological_clues": [],
    "misdirection_clues": [],
    "timeline_contradiction_clues": []
  }}
  """

  config = {
      "response_mime_type": "application/json",
      "response_schema": {
          "type": "object",
          "properties": {
              "physical_clues": {
                  "type": "array",
                  "items": {"type": "string"}
              },
              "psychological_clues": {
                  "type": "array",
                  "items": {"type": "string"}
              },
              "misdirection_clues": {
                  "type": "array",
                  "items": {"type": "string"}
              },
              "timeline_contradiction_clues": {
                  "type": "array",
                  "items": {"type": "string"}
              }
          },
          "required": [
              "physical_clues",
              "psychological_clues",
              "misdirection_clues",
              "timeline_contradiction_clues"
          ]
      }
  }

  response = retry_generate_content(
      client_obj=client,
      model_name=model,
      contents=prompt,
      config=config
  )

  data = json.loads(response.text)
  return data

## Module 5: Iterative Detective Meta-Controller

In [7]:
def generate_detective(client, model, setting):
  world_context = json.dumps(setting, indent=2)
  prompt = f"""
You are designing the protagonist detective for a classic whodunit murder mystery.

WORLD CONTEXT:
{world_context}

Create a detective who will investigate a murder in this setting. They must be internally motivated.

Generate:
"name": Full name and any title or honorific.
"background": 2-3 sentences on professional history, expertise, and why they are present at this location.
"personality": Dominant traits, flaws, how they behave under pressure.
"method": Investigative style (aggressive interrogation, quiet observation, timeline reconstruction, reading emotional tells, etc.).
"objective": Single primary goal — specific, not just "solve the murder".
"personal_stakes": Why THIS detective personally cannot afford to fail. Must be concrete.

Note: Ensure that the name of characters are uncommon and not classic names
like Alaistar Finch

Return ONLY a valid JSON object.
"""
  config = {
    "response_mime_type": "application/json",
    "response_schema": {
      "type": "object",
      "properties": {
        "name":            {"type": "string"},
        "background":      {"type": "string"},
        "personality":     {"type": "string"},
        "method":          {"type": "string"},
        "objective":       {"type": "string"},
        "personal_stakes": {"type": "string"}
      },
      "required": ["name", "background", "personality", "method", "objective", "personal_stakes"]
    }
  }
  response = retry_generate_content(client_obj=client, model_name=model, contents=prompt, config=config)
  return json.loads(response.text)


def generate_suspense_stakes(client, model, setting, crime, suspects):
  world_context    = json.dumps(setting, indent=2)
  crime_context    = json.dumps(crime, indent=2)
  suspects_context = json.dumps(suspects, indent=2)
  prompt = f"""
You are designing the suspense engine for a whodunit murder mystery.

WORLD CONTEXT:
{world_context}

CRIME SUMMARY:
{crime_context}

SUSPECTS:
{suspects_context}

Design the external pressure that makes the detective's failure catastrophic.

"countdown_mechanism": A concrete ticking-clock event if the detective fails. Be specific.
"countdown_deadline": When does it expire? In story-time terms.
"secondary_target": Full name of one innocent suspect the killer must silence next.
"secondary_target_reason": Why the killer needs to eliminate this person.
"escalating_events": Exactly 3 events that progressively worsen the detective's situation. Each has:
  trigger, event, effect_on_investigation

Return ONLY a valid JSON object.
"""
  config = {
    "response_mime_type": "application/json",
    "response_schema": {
      "type": "object",
      "properties": {
        "countdown_mechanism":     {"type": "string"},
        "countdown_deadline":      {"type": "string"},
        "secondary_target":        {"type": "string"},
        "secondary_target_reason": {"type": "string"},
        "escalating_events": {
          "type": "array",
          "items": {
            "type": "object",
            "properties": {
              "trigger":                 {"type": "string"},
              "event":                   {"type": "string"},
              "effect_on_investigation": {"type": "string"}
            },
            "required": ["trigger", "event", "effect_on_investigation"]
          }
        }
      },
      "required": ["countdown_mechanism", "countdown_deadline", "secondary_target",
                   "secondary_target_reason", "escalating_events"]
    }
  }
  response = retry_generate_content(client_obj=client, model_name=model, contents=prompt, config=config)
  return json.loads(response.text)


def detective_action_step(client, model, setting, detective, suspects, clues, stakes,
                          actions_so_far, plot_points_so_far, iteration_number):
  world_context     = json.dumps(setting, indent=2)
  detective_context = json.dumps(detective, indent=2)
  clues_context     = json.dumps(clues, indent=2)

  suspects_safe = [
    {k: v for k, v in s.items() if k not in ("is_murderer", "private_secret")}
    for s in suspects["suspects"]
  ]
  suspects_context   = json.dumps(suspects_safe, indent=2)
  previously_accused = [step["wrong_accusation"]["accused_name"] for step in actions_so_far]
  actions_context    = json.dumps(actions_so_far, indent=2) if actions_so_far else "None yet."
  event_index        = min(iteration_number - 1, len(stakes["escalating_events"]) - 1)
  pressure_event     = json.dumps(stakes["escalating_events"][event_index], indent=2)

  prompt = f"""
You are writing one step in a detective investigation. The detective has NOT solved the crime yet.

WORLD CONTEXT:
{world_context}

DETECTIVE:
{detective_context}

SUSPECTS (public info only):
{suspects_context}

AVAILABLE CLUES:
{clues_context}

SUSPENSE STAKES:
Countdown: {stakes["countdown_mechanism"]} — Deadline: {stakes["countdown_deadline"]}
Secondary target at risk: {stakes["secondary_target"]} — Reason: {stakes["secondary_target_reason"]}
Current pressure event: {pressure_event}

PREVIOUS ACTIONS (do NOT repeat):
{actions_context}

ALREADY ACCUSED (do NOT accuse again): {previously_accused}
PLOT POINTS SO FAR: {plot_points_so_far}
CURRENT ITERATION: {iteration_number}

The detective takes ONE new investigative action not yet taken.

STRICT RULES:
1. "is_correct" MUST be false. The detective does NOT solve the crime here.
2. Accuse a suspect NOT in the already-accused list and NOT the actual murderer.
3. "result" must explain specifically WHY the accusation failed.
4. "plot_points_added" must have atleast 4 distinct story beats this step generates.
5. "new_evidence_uncovered" must introduce information not yet mentioned.
6. "emotional_state" must reflect the detective's growing pressure or doubt.
7. Reference at least one prior action to show the investigation is building.

Return ONLY a valid JSON object.
"""
  config = {
    "response_mime_type": "application/json",
    "response_schema": {
      "type": "object",
      "properties": {
        "action_taken":           {"type": "string"},
        "action_target":          {"type": "string"},
        "result":                 {"type": "string"},
        "new_evidence_uncovered": {"type": "string"},
        "deduction":              {"type": "string"},
        "wrong_accusation": {
          "type": "object",
          "properties": {
            "accused_name": {"type": "string"},
            "reason":       {"type": "string"},
            "why_wrong":    {"type": "string"}
          },
          "required": ["accused_name", "reason", "why_wrong"]
        },
        "plot_points_added": {"type": "array", "items": {"type": "string"}},
        "emotional_state":   {"type": "string"},
        "is_correct":        {"type": "boolean"}
      },
      "required": ["action_taken", "action_target", "result", "new_evidence_uncovered",
                   "deduction", "wrong_accusation", "plot_points_added",
                   "emotional_state", "is_correct"]
    }
  }
  for attempt in range(3):
    response = retry_generate_content(client_obj=client, model_name=model, contents=prompt, config=config)
    step = json.loads(response.text)
    if not step["is_correct"] and len(step["plot_points_added"]) >= 3:
      return step
    print(f"  Action step attempt {attempt+1} failed validation — retrying")
  return step


def generate_mysterious_death(client, model, setting, crime, suspects, detective, actions_so_far):
  world_context     = json.dumps(setting, indent=2)
  crime_context     = json.dumps(crime, indent=2)
  suspects_context  = json.dumps(suspects, indent=2)
  detective_context = json.dumps(detective, indent=2)
  actions_context   = json.dumps(actions_so_far, indent=2)

  true_murderer_name    = next(s["name"] for s in suspects["suspects"] if s["is_murderer"])
  innocent_suspects     = [s for s in suspects["suspects"] if not s["is_murderer"]]
  accused_names         = [step["wrong_accusation"]["accused_name"] for step in actions_so_far]
  prev_accused_innocent = [s for s in innocent_suspects if s["name"] in accused_names]
  death_candidate       = prev_accused_innocent[0] if prev_accused_innocent else innocent_suspects[0]
  death_candidate_name  = death_candidate["name"]

  prompt = f"""
You are writing a pivotal event in a murder mystery: a second death occurs during the investigation.

WORLD CONTEXT:
{world_context}

ORIGINAL CRIME (full details):
{crime_context}

ALL SUSPECTS (with secrets):
{suspects_context}

DETECTIVE:
{detective_context}

DETECTIVE'S ACTIONS SO FAR:
{actions_context}

DESIGNATED VICTIM: {death_candidate_name}

This suspect must die now. Their death must:
1. Appear connected to the original murder (accident, suicide, or second murder).
2. Increase pressure and confusion for the detective.
3. Be fully explainable by the final epiphany (killer silenced them for what they knew).
4. Be consistent with the setting.

"victim_suspect_name": Must be "{death_candidate_name}".
"circumstances": How and where the body is found. Atmospheric and specific.
"apparent_cause": What the death looks like at first glance.
"true_connection_to_crime": Why the true murderer needed to eliminate this person.
"discovery_scene": 2-3 vivid prose sentences of the detective discovering the body.
"effect_on_suspects": How surviving suspects react.

Return ONLY a valid JSON object.
"""
  config = {
    "response_mime_type": "application/json",
    "response_schema": {
      "type": "object",
      "properties": {
        "victim_suspect_name":      {"type": "string"},
        "circumstances":            {"type": "string"},
        "apparent_cause":           {"type": "string"},
        "true_connection_to_crime": {"type": "string"},
        "discovery_scene":          {"type": "string"},
        "effect_on_suspects":       {"type": "string"}
      },
      "required": ["victim_suspect_name", "circumstances", "apparent_cause",
                   "true_connection_to_crime", "discovery_scene", "effect_on_suspects"]
    }
  }
  for attempt in range(3):
    response = retry_generate_content(client_obj=client, model_name=model, contents=prompt, config=config)
    death = json.loads(response.text)
    if death["victim_suspect_name"] != true_murderer_name:
      return death
    print(f"  Mysterious death attempt {attempt+1} killed the murderer — retrying")
  return death


def detective_breakthrough(client, model, setting, detective, suspects, clues,
                           crime_without_truth, actions_so_far, mysterious_death, stakes):
  world_context            = json.dumps(setting, indent=2)
  detective_context        = json.dumps(detective, indent=2)
  suspects_context         = json.dumps(suspects, indent=2)
  clues_context            = json.dumps(clues, indent=2)
  crime_partial_context    = json.dumps(crime_without_truth, indent=2)
  actions_context          = json.dumps(actions_so_far, indent=2)
  mysterious_death_context = json.dumps(mysterious_death, indent=2)
  wrong_accusations        = [step["wrong_accusation"]["accused_name"] for step in actions_so_far]

  prompt = f"""
You are writing the climax of a murder mystery. After many failed attempts, the detective has an epiphany.

WORLD CONTEXT:
{world_context}

DETECTIVE:
{detective_context}

SUSPECTS (with all secrets — use these to derive the answer):
{suspects_context}

AVAILABLE CLUES:
{clues_context}

CRIME DETAILS (partial — murderer and true motive not given; derive from suspect secrets and clues):
{crime_partial_context}

ALL DETECTIVE ACTIONS AND WRONG ACCUSATIONS:
{actions_context}

MYSTERIOUS DEATH:
{mysterious_death_context}

COUNTDOWN: {stakes["countdown_mechanism"]} — {stakes["countdown_deadline"]}
WRONG ACCUSATIONS MADE: {wrong_accusations}

"triggering_observation": One detail noticed earlier but dismissed, that now clicks. Must trace to a specific clue or action result.
"epiphany_reasoning": Full chain of logic from triggering observation to correct identification.
"correct_murderer_name": The true murderer (suspect where is_murderer=true in suspects list).
"evidence_chain": 4-6 specific pieces of evidence in order, each referencing a clue or prior action.
"explanation_of_mysterious_death": How does the epiphany explain who killed the second victim and why?
"confrontation_scene": 3-5 sentences of vivid prose — detective confronts murderer, killer reacts, countdown plays in.
"how_countdown_was_beaten": One sentence: how identification prevents the dire fate.

Return ONLY a valid JSON object.
"""
  config = {
    "response_mime_type": "application/json",
    "response_schema": {
      "type": "object",
      "properties": {
        "triggering_observation":          {"type": "string"},
        "epiphany_reasoning":              {"type": "string"},
        "correct_murderer_name":           {"type": "string"},
        "evidence_chain":                  {"type": "array", "items": {"type": "string"}},
        "explanation_of_mysterious_death": {"type": "string"},
        "confrontation_scene":             {"type": "string"},
        "how_countdown_was_beaten":        {"type": "string"}
      },
      "required": ["triggering_observation", "epiphany_reasoning", "correct_murderer_name",
                   "evidence_chain", "explanation_of_mysterious_death",
                   "confrontation_scene", "how_countdown_was_beaten"]
    }
  }
  response = retry_generate_content(client_obj=client, model_name=model, contents=prompt, config=config)
  return json.loads(response.text)

Coherence check: validates that breakthrough identifies correct murderer and explains the mysterious death

In [8]:
def check_coherence(client, model, crime, actions_so_far, breakthrough, mysterious_death):
  crime_context            = json.dumps(crime, indent=2)
  actions_context          = json.dumps(actions_so_far, indent=2)
  breakthrough_context     = json.dumps(breakthrough, indent=2)
  mysterious_death_context = json.dumps(mysterious_death, indent=2)

  prompt = f"""
You are verifying logical consistency in a murder mystery narrative.

CRIME (ground truth):
{crime_context}

DETECTIVE ITERATIVE ACTIONS:
{actions_context}

BREAKTHROUGH / FINAL ACCUSATION:
{breakthrough_context}

MYSTERIOUS DEATH:
{mysterious_death_context}

Answer "Yes" ONLY if ALL are true:
1. Final accused murderer matches actual murderer in CRIME.
2. Mysterious death is logically explained by breakthrough.
3. Physical clues support the solution.
4. No contradictions between detective actions and final explanation.
5. Breakthrough reasoning is internally consistent.

Also report:
- murderer_correctly_identified: true/false
- mysterious_death_explained: true/false
- wrong_accusation_count: total wrong accusations across all iterative steps
- inconsistencies: list specific logical contradictions (empty list if none)

Return ONLY a valid JSON object.
"""
  config = {
    "response_mime_type": "application/json",
    "response_schema": {
      "type": "object",
      "properties": {
        "answer": {"type": "string", "enum": ["Yes", "No"]},
        "murderer_correctly_identified": {"type": "boolean"},
        "mysterious_death_explained":    {"type": "boolean"},
        "wrong_accusation_count":        {"type": "integer"},
        "inconsistencies":               {"type": "array", "items": {"type": "string"}}
      },
      "required": ["answer", "murderer_correctly_identified", "mysterious_death_explained",
                   "wrong_accusation_count", "inconsistencies"]
    }
  }
  # All other models seem to take a lot of time for checking coherence and check for unnecessary things
  response = retry_generate_content(client_obj=client, model_name="models/gemini-flash-lite-latest", contents=prompt, config=config)
  return json.loads(response.text)

## Putting it all together

In [9]:
setting = generate_setting(client, model)
setting

{'setting': 'A subterranean seed vault located deep beneath the Svalbard permafrost, transformed into a high-security bio-repository.',
 'tone': 'psychological',
 'time_period': 'The mid-24th century, following the Resurgence Era.',
 'constraints': ['There are eight suspects remaining in the vault: Zephyrine Varl, Kaelen Nox, Xylia Mordant, Jaxon Thorne, Elara Vance, Soren Krow, Thistle Vyne, and Oryn Pax.',
  'All heavy-duty blast doors are locked into a fail-safe lockdown mode, accessible only by a single master keycard that has gone missing.',
  'An external glacial collapse has buried the primary exit and severed the fiber-optic link to the surface world.',
  'The case must be solved using forensic analysis of micro-residue on thermal suits, corrupted biometric entry logs, and psychological inconsistencies in suspect testimonials during the mandatory debriefing.']}

In [10]:
crime = generate_crime(client, model, setting)
crime

{'murderer': 'Elara Vance',
 'victim': 'Oryn Pax',
 'true_motive': "Oryn Pax had discovered encrypted data in the vault's core proving Elara's family was responsible for the genetic blight that caused the initial Great Famine, a secret she killed to prevent from reaching the Resurgence council.",
 'superficial_motive': "A heated argument over the dwindling oxygen supplies and Oryn's refusal to grant Elara access to the emergency seed stores for 'research'.",
 'means_of_murder': "Induced anaphylactic shock via a concentrated aerosolized extract of the extinct Phaseolus Primus plant, injected directly into Oryn's thermal suit oxygen intake valve.",
 'timeline': ['18:00 - The external glacial collapse triggers the fail-safe lockdown and severs communication.',
  "18:20 - Elara Vance steals the master keycard from Oryn's desk during the initial confusion.",
  '18:45 - Elara uses the master keycard to bypass the biometric log in the maintenance tunnel, heading to the thermal suit bay.',
  "

In [11]:
suspects = generate_characters(client, model, setting, crime)
suspects

{'suspects': [{'name': 'Elara Vance',
   'relation_to_victim': 'Research partner and professional rival.',
   'public_image': 'A methodical systems analyst known for her unflappable demeanor and technical precision.',
   'private_secret': 'She has been siphoning classified genetic blueprints to a black-market syndicate to settle a catastrophic family debt.',
   'possible_motive': 'Oryn discovered the data breach and intended to report her as soon as the communications link was restored.',
   'alibi': 'She claims she was recalibrating the oxygen sensors in Sector 4, which conveniently has a failed biometric scanner due to the collapse.',
   'is_murderer': True},
  {'name': 'Zephyrine Varl',
   'relation_to_victim': "Vault security officer and Oryn's former protégé.",
   'public_image': "A stern, hyper-focused guardian fiercely protective of the bio-repository's integrity.",
   'private_secret': 'She suffers from an early-onset neuro-degenerative condition that causes blackouts and memor

In [12]:
clues = generate_clues(client, model, setting, crime, suspects)
clues

{'physical_clues': ["Micro-residue of permafrost silica dust found on the soles of Elara Vance's boots, a substance only accessible via the unsealed ventilation shafts.",
  "Traces of aerosolized Phaseolus Primus pollen—an extinct botanical sample—recovered from the internal filtration membrane of Oryn Pax's thermal suit.",
  'A hollowed-out ancient grain sample canister in the vault core that appears slightly misaligned with its inventory tag.',
  'The master keycard is absent from the victim’s desk despite the room being locked from the inside during the collapse.',
  'A localized atmospheric pressure drop recorded in the maintenance tunnel at 18:45, indicating the manual opening of a bypass hatch.'],
 'psychological_clues': ['During the debriefing, Elara Vance provided a highly technical description of anaphylactic respiratory failure before the forensic analysis of the suit was conducted.',
  "Elara exhibits a 'hyper-fixation' on the integrity of the genetic archives, repeatedly re

In [13]:
DEATH_INJECTION_ITER  = random.randint(1,3) # Can be any iter from 1 to 3 to make it more interesting

# ── Step A: Generate detective persona and suspense stakes ──────────────────────
print("Generating detective persona...")
detective = generate_detective(client, model, setting)
print(f"Detective: {detective['name']}")
print(f"Objective: {detective['objective']}")
print(f"Personal stakes: {detective['personal_stakes']}")

print("\nGenerating suspense stakes...")
stakes = generate_suspense_stakes(client, model, setting, crime, suspects)
print(f"Countdown: {stakes['countdown_mechanism']}")
print(f"Deadline: {stakes['countdown_deadline']}")
print(f"Secondary target at risk: {stakes['secondary_target']}")

# ── Step B: Iterative action loop ──────────────────────────────────────────────
actions_so_far    = []
plot_points_count = 0
wrong_accus_count = 0
mysterious_death  = None
iteration         = 0
true_murderer_name = next(s["name"] for s in suspects["suspects"] if s["is_murderer"])

print("\n--- ITERATIVE INVESTIGATION LOOP ---")
while True:
    iteration += 1
    print(f"\nIteration {iteration}: generating detective action step...")
    step = detective_action_step(
        client, model, setting, detective, suspects, clues, stakes,
        actions_so_far, plot_points_count, iteration
    )
    actions_so_far.append(step)
    plot_points_count += len(step["plot_points_added"])
    wrong_accus_count += 1
    print(f"  Action: {step['action_taken'][:70]}...")
    print(f"  Wrong accusation: {step['wrong_accusation']['accused_name']} — {step['wrong_accusation']['why_wrong'][:60]}...")
    print(f"  Plot points added: {len(step['plot_points_added'])} | Total: {plot_points_count}")

    if iteration == DEATH_INJECTION_ITER and mysterious_death is None:
        print(f"\n  [Injecting mysterious death after iteration {DEATH_INJECTION_ITER}]")
        mysterious_death = generate_mysterious_death(
            client, model, setting, crime, suspects, detective, actions_so_far
        )
        print(f"  Victim: {mysterious_death['victim_suspect_name']} — Apparent cause: {mysterious_death['apparent_cause']}")

    if (iteration >= MIN_ITERATIONS and wrong_accus_count >= MIN_WRONG_ACCUS and plot_points_count >= MIN_PLOT_POINTS):
        print(f"\nLoop complete: {iteration} iterations | {wrong_accus_count} wrong accusations | {plot_points_count} plot points")
        break

# ── Step C: Detective breakthrough ─────────────────────────────────────────────
print("\nGenerating detective breakthrough...")
crime_without_truth = {k: v for k, v in crime.items()
                       if k not in ("murderer", "true_motive", "mistake_made")}
breakthrough = detective_breakthrough(
    client, model, setting, detective, suspects, clues,
    crime_without_truth, actions_so_far, mysterious_death, stakes
)

# ── Step D: Coherence check with retry ─────────────────────────────────────────
print("\nChecking coherence...")
for attempt in range(MAX_COHERENCE_RETRIES):
    coherence_result = check_coherence(
        client, model, crime, actions_so_far, breakthrough, mysterious_death
    )
    print(f"  Attempt {attempt+1}: {coherence_result['answer']} | Murderer correct: {coherence_result['murderer_correctly_identified']} | Death explained: {coherence_result['mysterious_death_explained']}")
    if coherence_result["answer"] == "Yes":
        break
    if coherence_result.get("inconsistencies"):
        print(f"  Inconsistencies: {coherence_result['inconsistencies']}")
    breakthrough = detective_breakthrough(
        client, model, setting, detective, suspects, clues,
        crime_without_truth, actions_so_far, mysterious_death, stakes
    )

print(f"\n{'='*50}")
print(f"FINAL STATS")
print(f"Coherence: {coherence_result['answer']}")
print(f"Iterations: {iteration} | Wrong accusations: {wrong_accus_count} | Plot points: {plot_points_count}")
print(f"Murderer identified: {coherence_result.get('murderer_correctly_identified')} | Death explained: {coherence_result.get('mysterious_death_explained')}")
print(f"{'='*50}")


Generating detective persona...
Detective: Magister Vaelen Voss
Objective: Locate the missing master keycard and neutralize the threat to the vault environmental seals before the backup power for the seed cryo-chambers reaches critical depletion.
Personal stakes: Voss is an unregistered synthetic hybrid whose stability depends on a specific neural-dampening serum locked within the high-security medical bay; without the master keycard to retrieve it, his neural pathways will permanently liquefy within seventy-two hours.

Generating suspense stakes...
Countdown: The vault's automated 'End-of-Cycle Sterilization' protocol, which floods all inhabited chambers with pure nitrogen to preserve the genetic samples if the master keycard is not re-synced with the central console following a high-security lockdown.
Deadline: 02:15 AM, exactly six hours after the discovery of Oryn Pax's body.
Secondary target at risk: Zephyrine Varl

--- ITERATIVE INVESTIGATION LOOP ---

Iteration 1: generating det

## Combining the entire story

In [14]:
world_context            = json.dumps(setting, indent=2)
detective_context        = json.dumps(detective, indent=2)
stakes_context           = json.dumps(stakes, indent=2)
clues_context            = json.dumps(clues, indent=2)
suspects_context         = json.dumps(suspects, indent=2)
crime_context            = json.dumps(crime, indent=2)
mysterious_death_context = json.dumps(mysterious_death, indent=2)
breakthrough_context     = json.dumps(breakthrough, indent=2)
coherence_context        = json.dumps(coherence_result, indent=2)

# Condense actions_so_far to save tokens
actions_summary = "\n".join([
    f"Step {i+1}: {s['action_taken']} → accused {s['wrong_accusation']['accused_name']}, "
    f"failed because: {s['wrong_accusation']['why_wrong']}. "
    f"New evidence: {s['new_evidence_uncovered']}. "
    f"Detective state: {s['emotional_state']}. "
    f"Plot beats: {'; '.join(s['plot_points_added'])}"
    for i, s in enumerate(actions_so_far)
])

final_prompt = f"""
WORLD CONTEXT:
{world_context}

DETECTIVE PERSONA:
{detective_context}

SUSPENSE STAKES:
{stakes_context}

AVAILABLE CLUES:
{clues_context}

SUSPECTS AND SECRETS:
{suspects_context}

CRIME DETAILS:
{crime_context}

ITERATIVE DETECTIVE ACTIONS (condensed):
{actions_summary}

MYSTERIOUS DEATH:
{mysterious_death_context}

DETECTIVE BREAKTHROUGH:
{breakthrough_context}

TASK:
You are the author of a classic whodunit murder mystery novel. Write the ENTIRE story as a continuous narrative in proper prose using full paragraphs. Do NOT use numbered lists.

STRUCTURE:
1. Open with the discovery of the original body — atmospheric, specific to the setting.
2. Introduce the detective by name within the first scene, weaving in their background, personality, method, and personal stakes naturally.
3. Use non-linear storytelling — flashbacks to character secrets, foreshadowing, shifting perspectives.

SUSPENSE:
4. Reference the countdown mechanism at least 3 times: early (reader learns the deadline), mid-story (deadline approaches), at climax (detective races to beat it).
5. The secondary target's danger must feel real — name them specifically as being at risk.
6. Each of the 3 escalating pressure events must appear at the appropriate narrative moment.

INVESTIGATION:
7. Each detective action step must appear as a full scene in sequence. Wrong accusations must feel completely logical in the moment — the reader should briefly believe each before it collapses.
8. The detective's emotional arc must progress: confidence → self-doubt → desperation → resolve.

MYSTERIOUS DEATH:
9. The mysterious death gets its own dramatic scene of at least 3 full paragraphs. Use the discovery_scene text as its core. Show how it shifts the atmosphere among survivors.

BREAKTHROUGH:
10. The epiphany scene must feel earned. Show the triggering observation clicking, walk through the full evidence chain, then write the confrontation scene with vivid detail.
11. End with a solution scene where the detective explains everything to the assembled suspects, resolving every clue and the mysterious death.

LENGTH AND QUALITY:
12. Minimum 2500 words. Rich descriptions, authentic dialogue, atmospheric detail. Professional whodunit quality.
13. Give the story a compelling title.

Output: Full story title followed by continuous prose paragraphs.
"""

response = client.models.generate_content(
    model=model,
    contents=final_prompt
)
print(response.text)



**The Permafrost Paradox**

The silence of the Svalbard Global Seed Vault was never truly silent. It was a pressurized, mechanical hum—the respiration of a dormant god. But at 20:15, that hum was punctured by the rhythmic, wet thud of a body hitting the reinforced glass of the Sector 4 airlock. Oryn Pax, the repository’s lead archivist, did not die with a scream. He died behind a visor, his fingers clawing at his own throat, leaving frantic, translucent streaks of condensation on the inside of his thermal suit’s faceplate. By the time the heavy-duty blast doors had finished their slow, hydraulic groan into a fail-safe lockdown, Oryn was a frozen statue of agony, and the only path to the surface was buried under three hundred million tons of collapsed glacial ice.

Magister Vaelen Voss stood before the airlock, his posture so unnervingly still he might have been mistaken for one of the vault’s high-security androids. As a forensic cryptographer for the Global Biosecurity Accord, Voss wa

## View and save the Generated Story



In [15]:
# Display the markdown story
from IPython.display import display, Markdown
display(Markdown(response.text))

**The Permafrost Paradox**

The silence of the Svalbard Global Seed Vault was never truly silent. It was a pressurized, mechanical hum—the respiration of a dormant god. But at 20:15, that hum was punctured by the rhythmic, wet thud of a body hitting the reinforced glass of the Sector 4 airlock. Oryn Pax, the repository’s lead archivist, did not die with a scream. He died behind a visor, his fingers clawing at his own throat, leaving frantic, translucent streaks of condensation on the inside of his thermal suit’s faceplate. By the time the heavy-duty blast doors had finished their slow, hydraulic groan into a fail-safe lockdown, Oryn was a frozen statue of agony, and the only path to the surface was buried under three hundred million tons of collapsed glacial ice.

Magister Vaelen Voss stood before the airlock, his posture so unnervingly still he might have been mistaken for one of the vault’s high-security androids. As a forensic cryptographer for the Global Biosecurity Accord, Voss was accustomed to death in sterile environments, but the Svalbard repository was different. This was a tomb of genetic memory, and now, it was a tomb for the living. Voss’s eyes, a flickering amber that betrayed his unregistered synthetic heritage, twitched beneath his brow. To Voss, the world was a cacophony of data. The high-pitched whine of the emergency lighting was a physical weight against his temples; the scent of recycled oxygen tasted like copper on his tongue. He reached into his pocket, fingering a small, empty vial. The neural-dampening serum he required to keep his synthetic processors from liquefying his organic brain was locked in the medical bay, three levels down. Without the master keycard—missing from Oryn’s cooling desk—Voss had seventy-two hours before his mind became a shattered mosaic of sensory feedback.

"He shouldn't have been in the suit yet," a voice cut through the hum. It was Elara Vance, the systems analyst. She stood with her arms crossed, her face a mask of clinical detachment that mirrored Voss’s own. "The perimeter check wasn't scheduled until 20:00. Oryn was always meticulous. He wouldn't skip a diagnostic."

Voss didn't look at her. His mind was already engaged in Chronological Discrepancy Mapping. He visualized the vault’s timeline as a series of overlapping translucent sheets. Layer one: the glacial collapse at 18:00. Layer two: the severing of the fiber-optic link. Layer three: the missing master keycard. "The lockdown is absolute," Voss said, his voice a low, melodic drone. "The 'End-of-Cycle Sterilization' protocol has been triggered by the lack of keycard synchronization. At 02:15 AM, this entire facility will be flooded with pure nitrogen to preserve the seeds. We are currently oxygen-rich. In six hours, we will be biological waste."

The remaining suspects stood in the flickering red glow of the corridor: Zephyrine Varl, the security officer whose hand hovered perpetually near an empty holster; Xylia Mordant, the head botanist with dirt under her fingernails and a sharp, defensive tongue; Kaelen Nox, the engineer who smelled of ozone and scorched copper; and Jaxon Thorne, the logistics manager whose jovial mask was beginning to fray at the edges. 

"We need that card," Zephyrine rasped, her eyes darting to the vents. "If we don't find it, the seeds stay safe, and we turn into cryogenic displays."

Voss’s first movement was toward the forensic lab. He needed the thermal suits. Using a portable micro-residue scanner, he began his audit. At 22:15, just as he was calibrating the sensors, a violent shudder rocked the vault. The glacial collapse wasn't over. A secondary structural shift ruptured a coolant line in the ceiling. A thick, white cryogenic fog poured into the lab, hissing like a nest of vipers. 

"Damn it!" Voss hissed, his hands trembling. The fog was more than an obstacle; it was a contaminant. The chemical coolant began to bond with the micro-residues on the suspects’ suits. He had to act fast before the physical evidence was erased. He turned his attention to Xylia Mordant. His scanner picked up a flare of activity: traces of a rare volcanic mineral, found only at the site of the airlock breach, were embedded in the fibers of her gloves. 

"Xylia," Voss said, the word a predatory strike. "You claim you were in the hydroponics bay during the collapse. But this mineral—basaltic silica—is only present in the primary exit corridor. You were with Oryn. You argued over the high-yield grain variants. You used your knowledge of botanical toxins to ensure he wouldn't report your illegal cultivation."

Xylia recoiled, her face turning a pale, sickly green. "I... I was there, yes. But I didn't kill him! We were working on a private project—a way out of this frozen hell. Oryn was going to recommend me for a promotion to the surface. Why would I kill my only ticket out?"

Voss’s internal processors whirred. He performed a temporal analysis of the mineral’s oxidation. His heart sank. The dust was "old"—it had been ground into the suit days ago, likely during a routine maintenance check they had performed together. The motive of professional jealousy collapsed under the weight of Xylia’s genuine terror. But more importantly, a new data point emerged. A partially melted micro-capacitor was found in the hydroponic bay’s intake vent. It wasn’t Svalbard hardware. It was a signal-dampener. Someone was spoofing the vault’s internal eyes.

The clock struck midnight. The vault’s central computer issued a hollow, synthesized chime. *Critical Integrity Wipe initiated. Purging non-essential biometric data to preserve power.*

"The logs," Kaelen Nox whispered, leaning against a bulkhead. "They're gone. We're blind."

Voss felt the first wave of neural decay. His vision pixelated, turning the red emergency lights into a rhythmic, blinding strobe. His serum level was at 70%. The air felt thick, like he was breathing through wool. He forced himself to focus on the 'Mass Displacement' alert that had flashed on a monitor just before the wipe. Sector 4. 18:45. 

"Kaelen," Voss barked, fighting the urge to clutch his head. "The displacement alert in the ventilation shaft. You're the only one who knows the conduits well enough to navigate them in total darkness. You were dodging the biometric scanners because you’d falsified the maintenance reports on the glacial shelf. Oryn found out. He was going to file a grievance that would end your career."

Kaelen didn't argue. He simply collapsed. He began to cough, a deep, rattling sound that produced gray carbon-soot on his lips. "I was... in the server cooling conduits," he gasped. "The fire... the collapse caused a short-circuit. I was trying to save the data. I wasn't in the vents, Voss. I was in the smoke."

Voss’s sensory processing disorder magnified the thumping of the vault’s life support until it sounded like a hammer against his skull. Kaelen’s soot-stained lungs were proof. He had been in the fire, not the vents. The detective felt a surge of synthetic vertigo. The murderer wasn't just hiding; they were using the vault’s own failures as a shroud.

At 01:15, the third escalation hit. A sweet, cloying scent began to drift through the common area’s ventilation. It was floral, ancient, and utterly terrifying. 

"Phaseolus Primus," Xylia whispered, her eyes widening in horror. "The extinct pollen. It’s being vented into the air."

The reaction was instantaneous. Jaxon Thorne began to claw at his chest, his breathing becoming a series of ragged whistles. Paranoia, a known side effect of the concentrated pollen, took hold. "It’s you, Voss!" Jaxon screamed, pointing a shaking finger. "You’re the one who’s not human! You’re killing us to protect the data!"

Voss ignored him, his eyes locked on a microscopic fragment of fabric caught in the latch of the Ancient Grain Archive’s secondary airlock. It was a scorched piece of a Level-5 bio-hazard lab coat, coated in the same iridescent pollen that was currently suffocating Jaxon. He looked at Jaxon’s logistics reports. The transmissions. Jaxon was a smuggler, yes, but his equipment was primitive—consumer-grade signal-shrouds that couldn't possibly have executed the 'Master Override' recorded at 18:45.

"It’s not Jaxon," Voss muttered, his voice barely audible over the screaming voices in his head. His serum level was at 55%. The ventilation hum had become a chorus of the damned. He turned to Zephyrine Varl. The security officer had been silent for too long. 

"Zephyrine," Voss said, stepping into her personal space. "Your biometric gap. Twenty minutes of 'stationary' status near the crime scene. You weren't on patrol. You were waiting for him."

Zephyrine didn't answer. She couldn't. She was staring at her empty holster, her eyes glazed. Voss realized with a jolt of clinical horror that she was in a dissociative fugue. Her neuro-degenerative condition had triggered a blackout. She wasn't the killer; she was a witness who couldn't remember what she’d seen. 

And then, the scream.

It came from the Hydroponics Bay. It was a high-pitched, warbling sound that was abruptly cut short. 

Voss ran, his movements jerky, his motor functions beginning to fail. He reached the Atmospheric Synthesis Chamber and stopped. 

Xylia Mordant was inside the sealed glass room. The chamber was designed for rapid botanical maturation, capable of flooding with concentrated nitrogen and CO2 at a moment's notice. The high-pitched whine of the nitrogen injectors pierced Voss's auditory filters like a serrated blade, vibrating through his synthetic marrow. Through the churning green mist of the synthesis chamber, Xylia’s face was pressed against the glass, her fingernails having left jagged, desperate scores in the frost-slicked surface before the air turned to poison. Her lifeless eyes stared directly at the Ancient Grain Archive across the hall, a silent accusation frozen in the sub-zero condensation.

The group’s collective composure shattered. Zephyrine Varl collapsed into a corner, sobbing incoherently. Jaxon was curled in a fetal position, gasping for air. 

"It was an accident," Elara Vance said, her voice steady, almost soothing. She stood by the control panel, her hands folded neatly. "The seismic tremors must have breached the seals. Kaelen’s negligence in maintenance has cost us another life."

Voss looked at Elara. Really looked at her. He saw the "mask of efficiency" she wore. And then, he remembered the sensor report. 

In his mind, Voss re-opened the digital file Elara had submitted for Sector 4. It was a masterpiece of technical jargon, but there were three basic mathematical errors in the oxygen-saturation ratios. At the time, he’d dismissed them as stress. But now, with his own neural pathways fraying, he saw the pattern. 

The errors weren't mistakes. They were a cognitive dissonance marker. 

"Your heart rate, Elara," Voss said, his voice dropping into an eerie, predatory stillness. "When we found Oryn’s body, everyone’s biometrics spiked. Jaxon’s hit 140. Zephyrine’s hit 150. Yours... yours stayed at a perfect 65 beats per minute."

Elara’s expression didn't change. "I was trained for crisis management, Magister. I don't panic."

"It’s not training," Voss countered, stepping toward her. "You’re using a neural-override program. A black-market 'Zen-State' loop. It keeps you calm so you can kill with clinical detachment. But it has a side effect. It throttles higher-order processing. It makes you bad at math."

He pulled up the coordinates from her "mistakes." He mapped them onto the floor plan of the Ancient Grain Archive. They pointed to a single, specific location: Grid 7-Alpha. 

"You didn't just kill Oryn because of an argument over oxygen," Voss said, his amber eyes glowing with a dying light. "He found the encrypted files. He found out that your family wasn't just a victim of the Great Famine. They were the architects of the genetic blight. You’re siphoning blueprints to pay off a debt, but you’re also trying to bury the truth of your bloodline."

The evidence chain solidified in the air between them, a ghost-map of guilt. "The permafrost silica on your boots—I found it before the coolant leak. It’s only in the ventilation shafts. You used them to reach the Phaseolus Primus samples. You sabotaged Oryn’s suit with a concentrated extract. You knew exactly what anaphylactic shock looked like because you’d seen the lab results before I even ran the forensic scan."

Elara’s eyes finally shifted. The mask didn't break, but the light behind it changed. It became cold. Sharp. "Oryn was a sentimentalist," she said, her voice like cracking ice. "He believed the truth was more important than the survival of the species. My family gave the world the tools to rebuild. What’s a little blight compared to the Resurgence?"

"And Xylia?" Voss asked.

"She was clever," Elara admitted. "She realized the mineral dust on her suit was a plant. She was going to tell you that the dust wasn't from the murder site, but from a maintenance bypass you hadn't checked yet. I couldn't let her finish that thought."

The vault’s central AI issued a final warning. *Warning. Nitrogen Sterilization in T-minus five minutes. 02:10 AM.*

Elara reached into her pocket and pulled out a small, hollowed-out grain canister. She didn't run. She simply stood there, a few feet from the archive terminal. "You're dying, Magister. I can see the static in your eyes. Even if you take this keycard, you won't have the strength to reach the medical bay."

Voss felt his legs give way. He slumped against the terminal, the world dissolving into a roar of white noise. His serum was at 40%. His motor functions were jerky, his vision a kaleidoscope of red and black. 

"Your pulse stayed flat because you weren't human, Vance; you were an algorithm of debt-repayment," Voss growled, the sound of his failing processors echoing like grinding metal in the small room. 

Elara took a step toward the ventilation grate. "Oryn was an obstacle to a clean slate. And in five minutes, the nitrogen will make sure his ethics die with the rest of you."

With a final, agonizing burst of synthetic adrenaline, Voss didn't go for Elara. He went for the canister. He lunged, his hand a blur of chrome and flesh, snatching the grain sample from her fingers. 

He didn't fight her. He slammed the master keycard into the archive terminal. 

"Override!" he screamed, the command echoing through the vault’s speakers. "Code Alpha-Zero-Niner! Abort sterilization!"

The vault groaned. The high-pitched whine of the nitrogen injectors, already beginning to hiss in the distance, sputtered and died. The red emergency lights flickered once, twice, and then turned a steady, calm blue.

*Sterilization aborted. Environmental seals restored. Medical Bay access granted.*

Voss fell to the floor, the master keycard still clutched in his hand. Elara stood over him, her face finally showing a flicker of something—not regret, but a profound, analytical disappointment. 

"You've doomed the repository, Magister," she whispered. "The blight will out."

"Maybe," Voss wheezed, his eyes closing as the sound of the medical bay doors sliding open echoed from below. "But you... you're just another data point now."

As the security drones—finally released from their lockdown—converged on the archive, Voss allowed the darkness to take him. He had beaten the clock by two seconds.

Later, in the sterile, white light of the medical bay, with the neural-dampening serum finally stabilizing his processors, Voss explained the final pieces to the survivors. He spoke of the 'Zero-Day' logic bomb Elara had uploaded at 18:15 to bypass the safety protocols. He showed them the fragment of the Level-5 lab coat, a silent testament to her presence in the restricted botanical stores. He explained how Xylia’s death had been the killer’s only true mistake—a desperate act that confirmed the technical nature of the murderer.

The permafrost of Svalbard remained, deep and unyielding. But inside the vault, the silence was finally real. The seeds were safe, the debt was exposed, and Magister Vaelen Voss, the synthetic ghost in the machine, had found the one thing his data couldn't predict: the sheer, messy weight of human justice.

In [17]:
output_filename = "generated_story.md"
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(response.text)
print(f"Story saved to {output_filename}")

Story saved to generated_story.md
